# MSC vs correlation (single patient)

Compare dense MSC and correlation matrices for one patient/band/phase.

**Legacy notebooks merged:**
- UTILS-FC_COMPARISON_PIPELINE.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
DATA_ROOT = Path('data/stereoeeg_patients')
patients = list_patients(DATA_ROOT)
assert patients, 'No patients found under data/stereoeeg_patients'

patient = patients[0]
phase = PHASE_LABELS[0]
band = BRAIN_BANDS_NAMES[2]

nperseg = 512
filter_time = 5000

corr = load_corr_matrix(patient, phase, band, filter_type='abs', zero_diagonal=True)
if corr is None:
    corr_res = compute_corr_matrix(
        patient,
        phase,
        band,
        filter_type='abs',
        zero_diagonal=True,
        filter_time=filter_time,
        cache_root=Path('data/corr_cache_dev'),
        verbose=True,
    )
    corr = corr_res.adjacency_matrix

msc = load_msc_matrix(patient, phase, band, nperseg=nperseg, sparsify='none', n_surrogates=0)
if msc is None:
    msc_res = compute_msc_matrix(
        patient,
        phase,
        band,
        nperseg=nperseg,
        sparsify='none',
        n_surrogates=0,
        filter_time=filter_time,
        cache_root=Path('data/msc_cache_dev'),
        verbose=True,
    )
    msc = msc_res.adjacency_matrix

corr.shape, msc.shape

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(corr, cmap='viridis')
axes[0].set_title('Correlation (abs)')
axes[1].imshow(msc, cmap='viridis')
axes[1].set_title('MSC dense')
plt.tight_layout()
plt.show()

tri = np.triu_indices_from(corr, k=1)
xc = corr[tri]
ym = msc[tri]

pearson = float(np.corrcoef(xc, ym)[0, 1])
spearman = float(spearmanr(xc, ym).correlation)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(xc, ym, s=4, alpha=0.4)
ax.set_xlabel('Correlation (abs)')
ax.set_ylabel('MSC')
ax.set_title(f'Pearson={pearson:.3f} Spearman={spearman:.3f}')
plt.tight_layout()
plt.show()